In [1]:
import sys
print(sys.executable)

c:\Users\DimiP\miniconda3\python.exe


# Homework 2 — Vector Search (LLM Zoomcamp 2026)

Solution du devoir du **Module 2 : Vector Search**.

| Question | Sujet |
|----------|-------|
| Q1 | Embedding d'une requete (ONNX) |
| Q2 | Similarite cosinus page vs requete |
| Q3 | Chunking + recherche vectorielle a la main |
| Q4 | VectorSearch avec minsearch |
| Q5 | Comparaison texte vs vectoriel |
| Q6 | Hybrid search (RRF) |

Cheatsheet : [`CHEATSHEET.md`](CHEATSHEET.md)

In [1]:
import numpy as np
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

COMMIT_ID = "8c1834d"
embedder = Embedder()

ModuleNotFoundError: No module named 'numpy'

## Q1 — Embedding d'une requete

In [ ]:
query_q1 = "How does approximate nearest neighbor search work?"
v = embedder.encode(query_q1)
print(f"Dimension : {len(v)}")
print(f"v[0] = {v[0]:.4f}")

## Chargement des lecons

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT_ID,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Pages : {len(documents)}")

## Q2 — Similarite cosinus

In [ ]:
target = "02-vector-search/lessons/07-sqlitesearch-vector.md"
page = next(d for d in documents if d["filename"] == target)
page_vec = embedder.encode(page["content"])
sim = float(page_vec.dot(v))
print(f"Cosine similarity = {sim:.4f}")

## Q3 — Chunking + recherche a la main

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
X = embedder.encode_batch([c["content"] for c in chunks])
scores = X.dot(v)
best_idx = int(np.argmax(scores))
print(f"Meilleur chunk : {chunks[best_idx]['filename']}")
print(f"Score : {scores[best_idx]:.4f}")

## Q4 — VectorSearch (minsearch)

In [ ]:
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

q4_query = "What metric do we use to evaluate a search engine?"
q4_vec = embedder.encode(q4_query)
q4_results = vindex.search(q4_vec, num_results=5)
print(f"1er resultat : {q4_results[0]['filename']}")

## Q5 — Texte vs vectoriel

In [ ]:
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

q5_query = "How do I store vectors in PostgreSQL?"
q5_vec = embedder.encode(q5_query)
vector_results = vindex.search(q5_vec, num_results=5)
text_results = text_index.search(q5_query, num_results=5)

vector_files = {r["filename"] for r in vector_results}
text_files = {r["filename"] for r in text_results}
only_vector = vector_files - text_files
print(f"Dans vector mais pas texte : {only_vector}")

## Q6 — Hybrid search (RRF)

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q6_query = "How do I give the model access to tools?"
q6_vec = embedder.encode(q6_query)
q6_hybrid = rrf([
    vindex.search(q6_vec, num_results=5),
    text_index.search(q6_query, num_results=5),
])
print(f"1er apres RRF : {q6_hybrid[0]['filename']}")

## Recapitulatif des reponses

| Q | Reponse |
|---|---------|
| Q1 | **-0.02** |
| Q2 | **0.37** |
| Q3 | **`02-vector-search/lessons/07-sqlitesearch-vector.md`** |
| Q4 | **`04-evaluation/lessons/05-search-metrics.md`** |
| Q5 | **`02-vector-search/lessons/08-pgvector.md`** |
| Q6 | **`01-agentic-rag/lessons/13-function-calling.md`** |